# TRADING INTELLIGENT · CH06 — LABORATOIRE A2
## **Défi autonome : construire, entraîner, comparer et choisir un modèle pour séries temporelles**

**Vous êtes l'ingénieur responsable.** Le professeur fournit les données fictives, une mission, quelques outils communs, des tests de conformité et des plages d'essai. **Vous écrivez** les fenêtres, le modèle MLP, une architecture récurrente RNN/GRU/LSTM, les comparaisons et l'analyse. Aucun modèle prêt à l'emploi dans STUDENT.

**Trois niveaux :** A — série **univariée**, B — comparaison et amélioration, C — série **multivariée financière** et décision avec coûts. Deux datasets distincts de celui du laboratoire guidé ; aucun accès réseau nécessaire.

**Navigation** : les cellules TODO sont **intentionnellement incomplètes** : l'exécution totale de STUDENT ne réussira qu'une fois votre implémentation terminée. Pour comprendre les mécanismes auparavant, exécuter LABO G1. MASTER contient les fonctions et les corrigés.

**Chronologie** : pas de mélange aléatoire de TRAIN/VAL/TEST pour les séries. Cibles matures à la date d'ajustement, paramètres de normalisation estimés uniquement sur TRAIN et test consulté **après** le choix final. Chaque résultat est à interpréter : il n'existe pas de « meilleur modèle universel ».

In [ ]:
import base64, zlib, io, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display
SEED=60624
np.random.seed(SEED);torch.manual_seed(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
plt.rcParams.update({'figure.figsize':(9,3.4),'axes.grid':True,'grid.alpha':.15})
print('CPU',torch.__version__,'| Pandas',pd.__version__)

def metrics(y,yhat):
    a=np.asarray(y).reshape(-1);b=np.asarray(yhat).reshape(-1)
    return {'MAE':float(mean_absolute_error(a,b)),'RMSE':float(np.sqrt(mean_squared_error(a,b)))}

def show_metrics(dic):
    return display(pd.DataFrame([{'Modèle':k,**v} for k,v in dic.items()]).sort_values('MAE').round(4))

def count_params(model):return sum(p.numel() for p in model.parameters())

def predict_numpy(model,X):
    model.eval()
    with torch.no_grad():return model(torch.as_tensor(X,dtype=torch.float32)).cpu().numpy()

def fit_torch(model,X_train,y_train,X_val,y_val,epochs=22,lr=.009,batch_size=64,patience=5):
    """Utilitaire commun fourni : un modèle est appris sur TRAIN ; early stopping sur VAL.
    Attention : le test ne doit jamais être passé à cette fonction.
    """
    Xtr=torch.as_tensor(X_train,dtype=torch.float32)
    ytr=torch.as_tensor(y_train,dtype=torch.float32)
    Xva=torch.as_tensor(X_val,dtype=torch.float32)
    yva=torch.as_tensor(y_val,dtype=torch.float32)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    best=float('inf');best_state=None;hist=[];wait=0
    for epoch in range(epochs):
        model.train()
        for ids in torch.randperm(len(Xtr)).split(batch_size):
            opt.zero_grad()
            pred=model(Xtr[ids]); loss=((pred-ytr[ids])**2).mean()
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            tr=float(((model(Xtr)-ytr)**2).mean())
            va=float(((model(Xva)-yva)**2).mean())
        hist.append((tr,va))
        if va<best-1e-8:
            best=va;best_state=copy.deepcopy(model.state_dict());wait=0
        else:
            wait+=1
        if wait>=patience:break
    model.load_state_dict(best_state)
    return model,np.asarray(hist,dtype=float)

# Mission A — Série univariée : définir une question avant tout modèle

Nous voulons prévoir **le niveau du signal à un pas** `signal[t+1]`, à partir des `L` dernières valeurs du signal, connues à `t`.

**Regardez le signal avant de choisir L**. Il est fictif ; sa loi génératrice n'est volontairement pas donnée dans le laboratoire étudiant. Le but est de diagnostiquer un problème, pas d'utiliser une formule secrète.

In [ ]:
# DATASET INTÉGRÉ : fallback exact, autonome et identique au CSV fourni.
# L'intégration compressée empêche de devoir télécharger un fichier dans Colab.
uni_B64 = (
    'eNpVXNvObqcNvM+z7D8C25weJ1KjqlLVm+T9lZllg90qN/9syjLgw9iY719//P3nr7/+8+///fHf36SJ/LT+0/RX+73nn/brB3+PBAb+vZW/5z'
    'egjdnbPCvxRbx1ky6yE++N88sx/JPNhDtgkbn7GssSFo6eYw8Z+c1OGds6c08bJ2EjLHZEp5U5FmEMtj3q6M2p9+myZZa5D+Gmaw07OVratxyd'
    'e0mfO/H+LX/qsXl2TiO+byKtq5okPojLVhWRXub5tlG6LnxiF/zbxn5MD5eQ+HZ8z65j6cM15JmmhnUFzn/65p+2dxuyEheXc6rpeMcsVIPv3D'
    'vFPzvxb11Y1drYo4Q/MbVrO31IGR5i4rPWW8GPf1a3dhn28E87Wjtt9z4l4U/61vbS01dP/Dvx2Y60YWUW6mjXs1SxQQlPwnawnmG51E8/cFCz'
    'W7My9/7Upi+stWyMUFFb3/O0Ngr87SOkGAq9yVnkU9VmS3aTMvwT/JhgGi2jP+PqZ5xmuXr5JFlbodr7whpHilkblvkUW3mkgG2fCWks4e9Em5'
    '0xIU3C9u0Vljn701/lgXI5YrLTHpXn+Rn1WnscTdiPs60BGVfO0psfc1MYmeZH/Tyljz3bLLArVz+yT+sJu80MXbCZOnvYTO+jpU0qD5QrPSJj'
    'nVbwTxmhRDanPuXVOFJoP9Sx51rjSGEWC6qei/2OFM6uwaPMlOY70s4PGqw74U8Xz9mmaxb48z5wYP1glxM+7pRw2iM3QD+f2aFGcHo5h4YCqF'
    'lf667HqBifR94Tjk0S/gSEFxF8NNFP5WRAo+F8E56fzsF+tpSZl6tWWzLf0i20Asejom8DLU4ZOzUWdmom7hu74Czaej7Q6NY53jpcFGw6cVcL'
    '7ZCxrZQx9AIeX5usMt7PGUEDRypFnuNOqsM3Yicf7r69L91mo8wTvv0cGs1I+UMxJo2gnTJ++HnwRPoo88+IBVPhIPJAwrczhu1dhm8/pwE/Xf'
    'b+U42GFWEzn8sYbu39IMbYeA5meBiHI9V5zrOw4ToAJbI2Z18JD4++8Ln9FJhKIFhL56YlTElkwx/Br+Xoz3cLHDTO7mnp8MhODjBpvQl/XmoN'
    'RJhRBOzhpUTOGTu/+flubO3ZZ1iuPkyde56+blwNmAZvj8CW+Inhgvg7z8MjulO/ZJXx7r3HpB8sG+DBHQpsCEkjpbkKMGzhw7lWme7WEQHTOw'
    '6eP9X3HDiNMvqz9gZ/2dbqKaMGQxoK+a+I04294SxUIE/Cn1qAB2GKnujnuhQbPhF3E/7kQ4geZ7YyesUcQs6T8KefYlSuUSbxvcUpj7MscQ/p'
    'g/TI1k7YDxo6u7vKSfzzUvCjiMY9lxkhvZlZWaVrxVHGi2dB82oFRIfzydFh5wobmnD1iQe7wIoQeUbibuemjI6njL86CiNfPfcrOB9oMCLeSN'
    'klxMHmzjK76ygUHQZ2ipSfnY81caiak6ir6Ja+LJ3IuizgYKEnHfgKotZ4eDCxmfgIVd8T3quMdyeF2GMwvZP4innAD0FhEt9OEKAHK2dxGj/p'
    'XZrm4M/YYXOqYNCJUhuVHur0OpgaAH2Gk7I6mhrwOXqQ2/LF/X1xgdbZs9zlNB6kQxjqcxJx25ob5zlzkaEB8Gk9yfeiAnwEYKiNpxcrWJptBI'
    'C1U263cuQNWFEvkyz321iLzZTbSR3I9YRhFNitCGzZJInXDjOHngzbz3J30HcEQKg09CVxV1GEdNFRhruGTvhc60+YHUrBPKtpe7LvS9+h5gaF'
    'LuOPb8wC8XjMc4epL+6MSYG/CKALDibpyw6lELozxNeEv1i0wEdh7PlJJ+84eeiF5oo+8g776dSuMonHUMbFvYskH73C/0BJJOf+zrlzLbYeT9'
    'nh/TEtTn+uhN2yNvKLvsskvodgkDiNk3vo0R96sSZUNGWJk0YWCCmLjEH2htDz5oo+998arV/fLCe4HhIABO8X/s9N6/BVeBct+PCDk800KOEZ'
    'Rg5Sk5nHucYPdsKstCe+Y01gXsnUzo0BZyHASMGD8YH+bJ1ScGd8YAdfIpv4FwRgd9jlntI74UM8RiIPTp546AaS2HPK8E9LDZtLD/PgzwfgrO'
    'eCNCmLZwFbGpl6TuK60Tc8V3uR54RuwNExOiQ6I/UC86xzrJsxgHWshD0HgGLM81L1cxUD1BsBtuDOCwy2hTgdZ9TbNfW2z2KFI/GohjRq40p4'
    'RBoIzrEvZSLu/v/Af4HxlumdpIJcwAPkNJHZrY1NhHkkHn4KIWpeB0PYFQAyMs+ciWtEI3DvKSl8lHJgBktXmWZFRIYhPX0hvsPxwE5PmeUzdz'
    'P83cvOuAIg3xlIhFKWTwHgG1iZyoV6WrfXRjopuc4vBoATwjBmK6M/xsysBsl9mdqZHvgyvFER5ItdSGwxWnPuYHqgNAg9F+5u6kiv+4K/1IRd'
    'QY8uKPlK2KmebMSuG857jySwQ4ugGmV08GiouOQO9ojyc8D991tJIRwbu8AJTs4SVE/ml+0m3INawPz7pRy9R15HvNuLxsTdk8KcoaaWwtx8Hy'
    'dx0gJ6aIVyA+YZuTOeBWAPx6g748Fe1mSFyHK4hM+Ea8Bhp5hB9waD/Sl75jUc+Cdw/Zup9B7sYIOPLk0ZPdxjJ5HZ9jK3VxsBg5DmHB4DkHSB'
    'BV3O3CWcPVTZTrvUsEtoAM6Oss+EnabA3kAmLeHpxQRkEv1tulyyjwyIwSHh7U6nM/fqCTtJha7AFNeDw8zh+5AwXLJIPKqJ8C3Wcj390hF8Fp'
    'uQeNARY54yy/h5fZE+qkvYq09Ga5GypOD7htRjrp3TB9sbSKNABHMfrwLAcqHYuQeX74M0sTKVuIsDB4XPWsHdmEB2mToXPOoSR3sasAThR8xZ'
    'u5W9DMKPeDyuYetXWH+xlKcyEvc9hm/46huJR7LamFM5u9Eo5Lv12YZ178SDr0ALoDblu8H5wPqijK6vno+Ij1S4jZzd48BpSJJDyzTr+YcOv8'
    '7x+SqYOzagp+Qe1rGFElUWfeV8OAcSJ02xnfHJ5x9bkcMVFYEBppSTeB6AQDJUWk7iJ41dHWSaPXHfQwXR1qUF9z2EK5Uz3do1q/kdrpe11lzR'
    'reYjyYLmpZR+1ghG8G3NUkyNqj1SHo0cUUs1H+xOwDRG4q7BYDZzzyfnq+Y33rvsVeaBnPN3MtM/f1qZxlMW7KbGHYJGMf+rB0AthrtZjVr+p+'
    '3f+PXgSAawMWK2E/a7G2j1mpqSeMBnPLlFeH2V/LbmElZaEg4fBlUfIwV01RAEsagqf6ifNe9zoHU5tYf7g5xnxUWURiHf029kGTvFdr4HD4CD'
    'WGW0e7A1wQ/XLPiK6gYodfhNjVL+55I2t2vczbq1fFbg4B6eveg9UIP7krPKeI1rJHwAbiZxV7zx3f60nrgrHnKnc927Rj3fS5Zgm/8nTxTK+u'
    'nwqfLwS/sG73N6jg8et+Vj5mW83VIsy3Yz8RH3PwalGQl//Flhw9jQnfB3rquTzpTJncfho6cYQZTzBWRwgCnlVjqRh2Wcs8sOeDV/gyfjvxTw'
    'C+7wO3Y5taNxlYj4JjN3Ny5o4MF2yz0Jvw6iAS+Qiwm/jvPHGfWc5FbzcXiIzXeVFpVc+apk74Simo+1Ix2MxFeznA+vBkqxZ8LToztmOM8N2q'
    'N9cINwPPnNfuuh0Iq98qNxDwcvi5NeO3GJbAD2sXP60IoGRwUnUKZ3reCV6Ym7EsdvrWkPnW9rbkG/4+jG0jLNifr/krGL9BH3QYkVtp3iRD2f'
    'KopYU/C4q11IFFrLz8ZdLVIlbP3zvhZZHmIeawC5CbfMBzfb1yrT7/BKmOMp9QgyN+fivc5I+FNTnPYGE18Ja5T5iUvCnxaYwNsfLXBwUN+ahH'
    'dIiClMy+Ru7QZFhQb3h4e1qwq8rCXs+2hnwd+/4xuhBVjoZidATh/GPuABbK0yT0RJqMBsJ7egR5Rsk1d1BQ81QCqSwfkW9eGnYAlpfyP5vIKM'
    'Pt88Lp2Dx4YvLOJEkAdPQCbac1m3qj91bZ1lfk/2YWwn0kiH4y508Gr7GcMt6/O2gxWshIOMIsHs44Wneeu6xsYKfZsfhf2vHAs9LsOH1xI+x3'
    'kS/vJRKJkynCX85aM4U2Y1kvB2n7Lh9meBj3u9yZ1/aFA5KpIsTfhzV+SsI3XglvWZcOFEUz6P71vpO0+Z2297EFaZXyd8vOiiZLMPvea+ycJ3'
    'jo48j6YLAjESj+s7RGGRWXANkjfICxP2yxvW7pMmzwgFcDDQrlZg19HNngTbuYdR1EHwxYI0x3s0ILuBT3pLXbeqA/cIY3q+7Zb14ZSxrqjsap'
    'T1ue3w7ghNmvC8tUpjUT7xcPyLvGg9ONovsJMtdyaq+qDc4DJr5+SuAaDhDDYnYfUSA5yDPa+5rgYIL5PKgjy2g90ioZkF9kaDs0CHdpk7NGB+'
    '+cyDb3cOmAN8eC4zPD40ESRBCx58/vD2feSKwtSRbt2UXqOy/9kiHBU2oOBRHm3IaluBI8+Dk9KVcFRwebU53+bua+g4arDfR/12BILJDpF+NG'
    'G/kkSO1/tzC/td9yjv33aZfXnNBGIjNdyJBxGFlY5dpz/h8sfNvDUq+xHJG8vkiXdXO2YKj7bvG+CZYCEMS+Jx+cRrrLSCKO639W1Ynd2bUA7S'
    'QmkpY6R6oP+na26BE/rB0kAv+3t5H5sQWoG9iwNWp1r20Ys65E4w9xRcLl+FzqRT3++osX45ZUG3hotsQdNX7Uv+Tiel15z/kj94E4SYO/7cvA'
    'A58IY+aeLREjHY4jAs8U9OKIGBnLz05Rb4ean2lYMSd/U4cGSg4ifx7b6T1FpnyhONO9pB0ufOefqNtR8hLfi7KjUsTRL3ZhmS4PXSghO3vDDW'
    'fhuAHP7qpAuJTksHH/X9/sXNcXKtnhYIPC3y6dwar+/ja0yZy2g3MzKEOVNwb9sZ7N9bZV+C+oEVw4xzOS8vgPstu+V5AXkMVllGRyz4ovhNDl'
    'lq9+tGVtues3oFfnZYfvEw8TCy/hXsCx4V/gmydWaZPm54bc5m118Tj4sosG7Ezxx/cz2Wz9vVVS/xf04VbAs6vxKXcB1zaJk+VEA3DuptPPEZ'
    'DaLrS7YT9x4jOMTZy6qipeNjf+0a7K3xN1Lu8apVt8Yvg/0vS/OjbvVDeanbc0me7UHbkZCu3HlXAuSca71M7Rb54Xzhl2aFPUs1+/jWg533zU'
    'GNltyWy/vgC+Eg7jdvlR/r1/086qvyk7o/ptxvzQ/5GHKpdVlIfzU8Zq+sBia+rtsYbARL3KMBrF3t0Ydb52c2ytPOz3oPr7Hc2laB/U5sspvJ'
    'cnIv7Wxy0P7UqL9YAHeqqRe36oe4t2TuMrl3abJ7Y1077VHIQy7Kok/P0be4owgzT1l6FHcQw0DwVn7SizsGarpOTwkjFtjCR2eBlzsSSLitfN'
    'IbIqbiMMpyosIPx7haWb1X+M+GKx23WNtvYY9nT44rift92OQNynMNEqYObcGRnlsC8Rq/5/Wd5bcyTVSmeQ2VR3TbdOGROsun5+H35qYvWJjm'
    '/JHpIdtny1iK7xqA5YOKtzKNuY8lYzllFjcv+J39uEaPxlvoObxC62W06yJUCIqXS/LqnjA3yw9GhR/8HinBTPmC+rM757auOR7sGQ7DWkoSZV'
    'xGgHlubuYFfhflsEsl4chPD47jJYq3wM8bl61RRrCvoO4FAJiiBV+xqO8zIndZPeoC9vryG2zla+BO2G4FZ7MWnHgwWZBt6LQm7jLCnpFw1K+e'
    'W8E5M3ogLcr7rjFmbJBI3IkAhsptU7Io8P94KARtnwmHOPAA7I9J3BM9yDik5da8tj4kF83K8B21M3zAZpHyRMMXmdXO3bktvDCzFQU7izq/59'
    'z46knhX9M+uzhWGX4bkDtyw/xqqAeiHTlbSu/xnUNhTyl8cD+webkt4pZF/jZBf8E6r/DPF4An71t3sCjyf3GTRPQUOMI+rImGk/i8VTINhmrZ'
    'sd8XIt6JOGulYx9sjX62SBMKwjvMuROPqM8T1LhysKjzf5vQeS37jvbd+clme5+cxH2P2VPDBvjEvcpCvhy1I4ddnN5YDlj5WXcHnQE0Wo4tiv'
    '2fgoAizGO5C1EjYoqhp8ziwsP379s857jvJY4JR6W5CZ4Iytdv1Xcuyj2CwB3C8mdKHxrC0hH88z3Dcg3AqnGBLUIlzuSdrN4TZyhq++2NPu8v'
    'm8nHSDwuAZB7gc8UfN9CGdLtp9/6iN5hM9pOcTzvh5vvez3r0cj7EaBAgp/aqzt/nuuUPcvkXzvf6ZvXiWX0R/UZcGRZ+eTnQtnpPuOpg70bgA'
    'aN37cfyN4NAN+tKHh6geWDYQfWn51Fh758HWGpAxqtHJu3RmU1n+vHtE3YFJhwBFDlzdEuc8drIlaU5A6PDv3GMuaJkprDcX1+2M76nLndDKDD'
    'Y88Uxq5ebO7VKNNE9fe7Linw9uRqMIKshE/UArclGAxP2IDwNtY83Ve2bFs0w1iU/8kq+ZCqiSTsnXxQf/iwlPqWfUEIdju5KT3aob/Jc/Hh89'
    'sigdYiomd/zInknAdfl2+8kt8Fj7uIyQZqs8Q/ZtVArI4V2W/RF7RNZhn9kdMlvEws5xkZXWuk2yNn0bihgpcdzwdG6Z+kTYY8O7ylf7qzI8+l'
    'jdfgxaAXNQl73fn4IoJe9PI5vG4vKxOXnXgkUIsdl1bwkB26eEmbvXZ+vtBghSS/Gq0cky20VoSPkg9LcO08Z3GL/+BstBgt03/STxb4V/2qX+'
    'GCnLFf9MFu6AP+WGcO9pSe6Tj8dIoYLXugbWJTE3bJtxho3kw4nKtuxIz6yeAge7KgVsZ/HgqOCHnuyTP12z4QTrn3xvaq/sKAD4aTZ6r+KGLA'
    '5uJ1nb3+fCxnnJ5zz2D/cNqrjVFGD0/DvouYMvpzXJiaaZgk/O3slxC2d27Rno9obMbXXgnfm76NUNRm4nHTx1Y+sYKHi+osO1nKeHu8oBhzPr'
    'byWvSFV7RraeIei/jOR84u8/v2soP8RcAZasG3lOvMnCT0gl2VVvYrTtpAfc/Ugg9vq6K3zC9KvDj57kjKJ1d0XB82vpfh99ERb0SeAqyw9Ym0'
    'TffzLytY/gD7t6i7WZT9P+Zo3wVgwlQAntA6TcpoCs6kdd9Cs73WfObP2vUUOK5xOvskU5JPAZRPKI/2HB01f4QKjZc1ljV/UrSTPj2a80HV5d'
    'AaE55hcizYl0mip5S9Bf35ihWHz3QWW7tG4v5wiw8UEVxz/cHejtk6L86vvMixewVlWfQXEIi5t+XsUduFdvIhYa4p6L2NDgW18tUgb8a7n5bz'
    'RBPP5LOF1XK10cSDPYdZv/lv4b/zdRmbDxKX29xDT7ITH9H4ddgjMBKPhi020Ma9qGXpn0/p+GZZE4+XBwcx6WV60dOvTLqxDzmLx/b9XQKnLP'
    'F043t01grsTzcW+WWd4yMCNkGld37QIwBZzch4v2/HFuYGjczR9/EO6xbxeM9e1R9UCke3NBfplR7eQalIGe0FCtGa3OywdTaV4v+RokSeP1nC'
    'PXlAN8+H5ba64a4YfDKN5H3nAbliYK9szPVc723SV7bi8jVV4hpvtcEz0k/don//XmTFIxDLoj97h3i5NhOPflm2us9RcDcn3rhJ1gbOLfjO0c'
    '8eRZ7IA3CoMJHnfU884ZtIQrVJLitCgdBJyErxI+IPJlujwNH0MdkG/YhZFP356mStZ8Tn5X30nHVNcQM4DBGiW64pfAH9PTQ45/G2bniCU/Tj'
    'lf15GzIsdzhigS2+SCgfXffSjYwwlxSegH2BZ97wzkp7vNU6vBiSgofFdz5suBwpK/9wGqA3s4yPls7DzqKrCNmrz55sfbGJeOT0zLZ2L/Ofez'
    's8plrO74oAD4Ske++cxxVBG3+h4KVK3q3vpIBPFloZHx5rKHzQLuPd8ln3GzPFieu+yVKQlNHRo3boJsd4eFj+5DPJlagrpbBvq4gY/XvQjX5T'
    '1tfcDxQMspfBXg6Er2kvrtx2fd6UQSdnGX0fvfC1dZklCv9QADZjJhzWZMoqQui7F/6/w57IuOSNj459xGYY2bxW1uN3OHjLdTL19cq/t4jzJz'
    'AKvLzDlqS6TnJzK3iUcTlBduzzNz1kjIK7ou4O+7A6XqOYxDeelxW+Dn/ePiOS3YQ2a/+yJ5+zpDw31+Ol/36GcLt4WdEZeaj9PdzmU2m5Ke3r'
    '8G+kf9J67kLwhckWf5ll/Lg/vkFTyyMJs6fXu37s1f/hzdm4nrJ7m9dgifCxrlv/b+waNbk86vbsUzhQmlt96xJtXuyGt8f0b/mfzRCgrmsm/N'
    'VQ9ueUpIym2MZkkZe0CVMShFT2WuyEnf8h2iIIPWOUuNlfgx2Ap8ASWRHnflv+OvzBfHAavYyPmygmVy9f6HIpIOLSmlX24P+8GM4y4OvxZ12d'
    'T6hzw1wFDEyXtxE5T1Tw6MrnWDlPlHgJ7zR5uSqwvpfblusKEsjU8LY5Wfb4M8viy8Nc73P9GzpfxnvzBy9BR5TvR94B8JcKVrwgGO8OAE6pMf'
    '7thD/WBe7aerj9ER37X0Z6YNr+yVF+gQeRvMd97Kg3AIvX4LNMfvvW+JMP7iNG/mAPTu9YZOSj/AYPfxpjtDLavRWSgFvtG1H//56okCpYmSTa'
    'UklD/FZglPI/M5UWj2NH+RGeQwPbOz/qkYDPQnjgCUd6sHix5UF+lC5/OBMZQfdGlv+VD/Is3nuP0rVvfEx9WplnRzhHUhIF/ZH1fxwz0qf4/Z'
    'ERFwBfwQbZSpsp5r0JniAjcec3StP++fqrZsL+Mx72NfVZws5c2DEW72NHVP+p1nyJIe+b0eEvvK1EijQe/J01zEjv1dOIX9/58h2Ydhv5Rad/'
    'x5ikvF2UlxpM3hcV2DxeLTDAgnoj4kGaHn1o4/32jiBUL1DslDq6v6BzI34FYETN3/M98O54LjpK0Z93rzOi2yhN+8L8rX41EkH6GBlFSA8Dyp'
    '9Uau845akFKP1IXX/PfBbrGhe9HfusqCLLOJr4fQMFTdnvmG9xX76eCTmSeDzVhXdhi1bil0Ybtcgefju3BjlODu/3OTHve1MaP1MWJu+rwfFK'
    '+6ybGd8HJhw/eWPk0GWSuGXGlo32PMn7/R12Xt57ifGK+3zsPVr8KMnI4v7iFWa6TL3PNKGKjc0H/wAOEGzb'
)
possible=[Path('CH06_A2_UNIVARIE.csv'),Path('datasets/CH06_A2_UNIVARIE.csv'),Path('../datasets/CH06_A2_UNIVARIE.csv')]
found=next((p for p in possible if p.exists()),None)
uni=pd.read_csv(found if found else io.BytesIO(zlib.decompress(base64.b64decode(uni_B64))))
uni['date']=pd.to_datetime(uni['date'])
print('Dataset uni:',len(uni), 'lignes; source',found if found else 'archive intégrée')
print(uni.head(3).to_string(index=False))

In [ ]:
assert uni['date'].is_monotonic_increasing and uni['signal'].notna().all()
plt.plot(uni['date'],uni['signal']);plt.title('Série univariée synthétique — observations chronologiques');plt.xlabel('Date');plt.ylabel('Signal');plt.show()
values=uni['signal'].to_numpy(dtype='float32')
T=len(values)
print('Nombre de dates T =',T,'| départ de la série :',uni.date.iloc[0].date())

### Exercice initial : dessinez le tableau supervisé

Pour `L=3`, `h=1` et une série de dix valeurs, expliquez pourquoi il existe `10-3-1+1=7` lignes. Pour chaque ligne, associez `X[t-L+1:t+1]` à `signal[t+h]`.

La fonction demandée doit renvoyer `X(N,L,1)`, `y(N,1)` et le tableau `decision_index(N,)` des dates de décision. Les indices de maturité de la cible seront `decision_index+h`.

### Mission · A1 — Construisez les fenêtres univariées

**Signature imposée :** `faire_fenetres(signal, L, h=1)`. Pour chaque ligne, l'entrée utilise uniquement les indices allant de `t-L+1` à `t` inclus, la cible est `signal[t+h]`. Retourner trois `numpy.ndarray` de types numériques, dimensions indiquées ci-dessus.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def faire_fenetres(signal,L,h=1):
    """X(N,L,1), y(N,1), decision_indices(N,) ; aucune fuite future."""
    raise NotImplementedError('A1 : construisez les fenêtres et la cible à t+h')

In [ ]:
tiny=np.arange(10,dtype='float32')
Xa,ya,ta=faire_fenetres(tiny,3,2)
assert Xa.shape==(6,3,1) and ya.shape==(6,1) and ta.shape==(6,)
assert np.array_equal(Xa[0,:,0],[0,1,2]) and ya[0,0]==4 and ta[0]==2
assert np.array_equal(Xa[-1,:,0],[5,6,7]) and ya[-1,0]==9
print('A1 VALIDÉ — première fenêtre',Xa[0,:,0],'cible',ya[0,0],'; dernière décision',ta[-1])

## A2 — Séparer TRAIN, VALIDATION, TEST **sans faire entrer un label immature**

Une prédiction faite après `t` ne peut apprendre avec une cible qui ne sera observable qu'après `t+h`. Si VALIDATION commence à la décision `cut_val`, les lignes TRAIN doivent satisfaire `target_index < cut_val`. Idem pour le passage vers TEST.

Ce découpage est **figé chronologiquement**. Le scaler et les hyperparamètres sont choisis sur TRAIN/VALIDATION uniquement.

### Mission · A2 — Construisez la séparation chronologique

**Contrat :** `split_indices(decision, h, cut_val, cut_test)` renvoie `itr, iva, ite` :
- TRAIN : `decision < cut_val` et `decision+h < cut_val`
- VALIDATION : `cut_val <= decision < cut_test` et `decision+h < cut_test`
- TEST : `decision >= cut_test`

Ces trois vecteurs contiennent des indices **de lignes de fenêtres**, pas des dates de calendrier.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def split_indices(decision,h,cut_val,cut_test):
    """indices TRAIN/VAL/TEST sur les lignes fenêtrées, cibles matures aux frontières."""
    raise NotImplementedError('A2 : appliquez les conditions temporelles sur decision et decision+h')

In [ ]:
d=np.arange(2,10)
itr0,iva0,ite0=split_indices(d,1,6,8)
assert np.array_equal(d[itr0],[2,3,4])
assert np.array_equal(d[iva0],[6])
assert np.array_equal(d[ite0],[8,9])
L,h=12,1
Xraw,yraw,decision=faire_fenetres(values,L,h)
cut_val=int(.60*T);cut_test=int(.80*T)
itr,iva,ite=split_indices(decision,h,cut_val,cut_test)
assert len(itr)>100 and len(iva)>50 and len(ite)>50
assert max(decision[itr]+h)<cut_val
assert max(decision[iva]+h)<cut_test
assert min(decision[iva])>=cut_val and min(decision[ite])>=cut_test
print('A2 VALIDÉ :',len(itr),'TRAIN,',len(iva),'VALIDATION,',len(ite),'TEST')

## A3 — Baseline et normalisation TRAIN

Pour une série de niveaux, la baseline raisonnable est **dernière valeur observée**. Il faudra faire mieux que cette référence, et pas seulement obtenir une MAE faible en valeur absolue.

La normalisation de la seule feature doit être estimée sur les lignes TRAIN ; appliquer la **même** transformation à VALIDATION et TEST. La cible reste sur son échelle initiale pour faciliter l'interprétation.

In [ ]:
scaler_uni=StandardScaler().fit(Xraw[itr].reshape(-1,1))
Xuni=scaler_uni.transform(Xraw.reshape(-1,1)).reshape(Xraw.shape).astype('float32')
assert Xuni.shape==Xraw.shape
base_val=Xraw[iva,-1,0].copy()
print('Baseline persistance VALIDATION',metrics(yraw[iva],base_val))
print('Entrée MLP',Xuni[itr].reshape(len(itr),-1).shape,'| RNN/GRU/LSTM',Xuni[itr].shape,'| y',yraw[itr].shape)
# Pas d'accès aux cibles TEST pour la sélection des architectures.

### Mission · A3 — Codez un MLP

**Constructeur :** `MLPReg(lookback,features,hidden,outputs=1)`. Il doit aplatir l'entrée `(B,L,F)` et produire `(B,outputs)` avec au moins une non-linéarité et des couches `nn.Linear`. N'incluez pas de sigmoïde pour la régression.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
class MLPReg(nn.Module):
    def __init__(self,lookback,features,hidden=16,outputs=1):
        super().__init__()
        raise NotImplementedError('A3 : définissez la couche cachée et la sortie')
    def forward(self,x):
        raise NotImplementedError('A3 : aplatissez et prédisez')

In [ ]:
test_model=MLPReg(12,1,16,1)
with torch.no_grad(): z=test_model(torch.zeros(7,12,1))
assert tuple(z.shape)==(7,1) and count_params(test_model)==225
print('A3 VALIDÉ, modèle MLP :',count_params(test_model),'paramètres ; sortie',z.shape)

### Mission · A4 — Codez un réseau à mémoire générique

**Constructeur :** `ReseauMemoire(kind, features, hidden, outputs=1)` pour `kind` parmi `RNN`, `GRU`, `LSTM` ; utiliser `batch_first=True`. Dans `forward`, exploiter la **dernière sortie temporelle** du bloc récurrent, puis une tête `nn.Linear` pour obtenir `(B,outputs)`. N'aplatissez pas le temps.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
class ReseauMemoire(nn.Module):
    def __init__(self,kind,features,hidden=12,outputs=1):
        super().__init__()
        raise NotImplementedError('A4 : créez nn.RNN/nn.GRU/nn.LSTM et la tête')
    def forward(self,x):
        raise NotImplementedError('A4 : récupérez la dernière sortie temporelle')

In [ ]:
for k in ['RNN','GRU','LSTM']:
    mod=ReseauMemoire(k,1,12,1)
    with torch.no_grad():out=mod(torch.zeros(7,12,1))
    assert out.shape==(7,1)
    print('A4 VALIDÉ',k,'params=',count_params(mod),'sortie=',tuple(out.shape))

## Mission B — Comparer, améliorer et **choisir avant le TEST**

Chaque étudiant lance un protocole limité et comparable : même TRAIN/VALIDATION, même cible, même `L`, même loss et mêmes métriques. MLP, RNN, GRU, LSTM sont comparés à la persistance et à Ridge. Une amélioration d'architecture est évaluée sur VAL, jamais choisie sur TEST.

**Budget raisonnable CPU :** 12–25 epochs, petites tailles cachées, early stopping. Vous pouvez proposer **une** expérience supplémentaire sur `L=6` ou `L=24`, mais toutes les fenêtres et le scaler doivent être reconstruits proprement pour la comparaison.

In [ ]:
# Utilitaire de référence linéaire, sans choix de modèle caché.
ridge=Ridge(alpha=8.).fit(Xuni[itr].reshape(len(itr),-1),yraw[itr].ravel())
val_predictions={'Persistance':base_val,'Ridge':ridge.predict(Xuni[iva].reshape(len(iva),-1))}
val_scores={k:metrics(yraw[iva],p) for k,p in val_predictions.items()}
print('Références sur VAL :');show_metrics(val_scores)

### Mission · B1 — Conduisez la comparaison complète

**À écrire :** créer et entraîner un MLP, un RNN, un GRU et un LSTM avec les fonctions précédentes. Paramètres communs initiaux `hidden=12`, `epochs=22`, `lr=.009`, `batch_size=64`. Conserver leurs prédictions VALIDATION, leurs courbes de perte et leurs modèles dans `models_uni` et `history_uni`, puis compléter `val_predictions`/`val_scores`.

**Contrainte :** interdiction de lire `yraw[ite]` dans cette cellule. Afficher le tableau VALIDATION, avec MAE, RMSE, paramètres et epochs ; ne pas décréter un vainqueur universel.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
models_uni={};history_uni={}
# TODO : construire, entraîner, mesurer MLP, RNN, GRU et LSTM.
# Variables attendues : models_uni, history_uni, val_predictions, val_scores.
raise NotImplementedError('B1 : entraînez les quatre architectures uniquement sur TRAIN et VALIDATION')

In [ ]:
assert set(['Persistance','Ridge','MLP','RNN','GRU','LSTM']).issubset(val_scores)
assert all(np.isfinite(v['MAE']) and np.isfinite(v['RMSE']) for v in val_scores.values())
assert set(models_uni)=={'MLP','RNN','GRU','LSTM'}
print('B1 VALIDÉ — six modèles comparés sur VALIDATION')

### Mission · B2 — Figez le meilleur modèle sur validation

**À produire :** une variable `chosen_uni` qui contient le nom du modèle ayant la plus petite **MAE VALIDATION**. Elle doit appartenir à `val_scores`, sans accéder aux données TEST. Expliquez aussi le nombre de paramètres et la complexité temporelle dans votre compte rendu.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
# TODO : déduire le nom du meilleur modèle depuis les scores VALIDATION.
raise NotImplementedError('B2 : choisissez chosen_uni avec la MAE de VALIDATION')

In [ ]:
assert chosen_uni in val_scores
print('B2 VALIDÉ — modèle choisi sans consulter TEST')

### Consultation unique du TEST — lecture terminale
**Attention :** ce bloc ne doit être exécuté **qu'après** avoir rempli les cellules précédentes, choisi le modèle et consigné votre justification. On ne revient pas modifier le choix après lecture de TEST.

Si le modèle choisi est une baseline, utilisez-la ; sinon, utilisez ses poids entraînés. La série est une expérience pédagogique sans garantie de généralisation.

In [ ]:
if chosen_uni=='Persistance':pred_test_uni=Xraw[ite,-1,0]
elif chosen_uni=='Ridge':pred_test_uni=ridge.predict(Xuni[ite].reshape(len(ite),-1))
else:pred_test_uni=predict_numpy(models_uni[chosen_uni],Xuni[ite]).reshape(-1)
print('UNIVARIÉ — TEST (une seule consultation) :',chosen_uni,metrics(yraw[ite],pred_test_uni))
plt.plot(uni.date.iloc[decision[ite]].iloc[:120],yraw[ite].ravel()[:120],label='Observé')
plt.plot(uni.date.iloc[decision[ite]].iloc[:120],pred_test_uni[:120],label='Prévu')
plt.title('Cible et prévision sur les premières dates TEST');plt.legend();plt.show()

## Mission B+ — Extension directe multi-horizon

Les **sorties multiples** ne signifient pas une séquence d'entrées plus longue. Si l'on veut prédire simultanément les niveaux des séances `t+1, t+2, t+3`, la cible devient un vecteur de longueur 3. Il faut également garantir que toutes ces cibles étaient connues avant les frontières TRAIN/VAL.

**À réaliser en exercice** : utilisez un MLP `outputs=3` et réentraînez-le sur un dataset de cibles trois pas. Puis comparez à une prévision récursive, en indiquant que les deux démarches ne répondent pas au même protocole.

### Mission · B3 — Construisez la cible multi-horizon directe

**Contrat :** `fenetres_multipas(signal,L,H)` renvoie `X(N,L,1)`, `Y(N,H)` et `decision(N,)`. La ligne `Y[i]` est `[signal[t+1],...,signal[t+H]]`. Le nombre de lignes vaut `T-L-H+1`. **Ne confondez pas** le paramètre `H` (nombre de sorties futures) avec `h` (date future d'une seule cible).

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def fenetres_multipas(signal,L,H):
    raise NotImplementedError('B3 : générez les H futures valeurs pour chaque fenêtre')

In [ ]:
a,b,c=fenetres_multipas(np.arange(10),3,2)
assert a.shape==(6,3,1) and b.shape==(6,2)
assert np.array_equal(b[0],[3,4]) and c[0]==2
print('B3 VALIDÉ :',a.shape,b.shape)
# Pour cette extension, reconstituer les dates matures : t+H aux frontières.

# Mission C — Vers un véritable problème financier multivarié

Maintenant nous passons à une table de cours fictifs. Les étudiants reçoivent les prix **open/close**, les volumes et un indicateur d'un second actif. Aucun signal d'avenir ni classe pré-calculée.

**Horloge** : après `close[t]`, les caractéristiques des séances au plus `t` sont observables. À `open[t+1]`, on peut prendre position ; le P&L est le mouvement `open[t+1] → close[t+1]`. Les « business days » simulés ne sont **pas** un calendrier boursier réel.

**Question cible** : prévoir le rendement intrajournalier logarithmique `100×log(close[t+1]/open[t+1])` (points de %) avec un `L=16` et `F=5` ; estimer d'abord les baselines.

In [ ]:
# DATASET INTÉGRÉ : fallback exact, autonome et identique au CSV fourni.
# L'intégration compressée empêche de devoir télécharger un fichier dans Colab.
market_B64 = (
    '...'
)
possible=[Path('CH06_A2_MARCHE_MULTI.csv'),Path('datasets/CH06_A2_MARCHE_MULTI.csv'),Path('../datasets/CH06_A2_MARCHE_MULTI.csv')]
found=next((p for p in possible if p.exists()),None)
market=pd.read_csv(found if found else io.BytesIO(zlib.decompress(base64.b64decode(market_B64))))
market['date']=pd.to_datetime(market['date'])
print('Dataset market:',len(market), 'lignes; source',found if found else 'archive intégrée')
print(market.head(3).to_string(index=False))

In [ ]:
assert market['date'].is_monotonic_increasing
assert market[['open','close','volume']].gt(0).all().all()
assert len(market)>800
fig,ax=plt.subplots(1,2,figsize=(10,3.4))
ax[0].plot(market.date,market.close);ax[0].set_title('Clôtures fictives')
ax[1].plot(market.date,market.volume);ax[1].set_title('Volumes fictifs')
plt.tight_layout();plt.show()
print(market.head(2).to_string(index=False))

### Mission · C1 — Fabriquez les caractéristiques causales

**Contrat :** `creer_features_financieres(df)` doit renvoyer une copie du tableau avec ces cinq colonnes nouvelles, toutes calculées à date `t` :
1. `r_cc_pct = 100×log(close[t]/close[t-1])`
2. `vol_rel = volume[t] / mean(volume[t-1],...,volume[t-5])`
3. `sigma5_pct = écart-type empirique (ddof=1) des 5 derniers r_cc_pct connus`
4. `mom5_pct = 100×log(close[t]/close[t-5])`
5. `b_ret_pct` déjà fournie : rendement du second actif, connu à la clôture t.

Ajouter `y_next_oc_pct = 100×log(close[t+1]/open[t+1])` uniquement comme **cible** ; ne jamais l'utiliser comme feature. Ne pas remplir les premiers `NaN` avec des informations futures.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def creer_features_financieres(df):
    raise NotImplementedError('C1 : calculez les cinq variables et la cible future distincte')

In [ ]:
feat_cols=['r_cc_pct','vol_rel','sigma5_pct','mom5_pct','b_ret_pct']
fin=creer_features_financieres(market)
assert set(feat_cols+['y_next_oc_pct']).issubset(fin.columns)
assert pd.isna(fin['y_next_oc_pct'].iloc[-1])
assert np.isclose(fin['r_cc_pct'].iloc[1],100*np.log(fin['close'].iloc[1]/fin['close'].iloc[0]))
assert np.isclose(fin['y_next_oc_pct'].iloc[10],100*np.log(fin['close'].iloc[11]/fin['open'].iloc[11]))
assert pd.isna(fin['vol_rel'].iloc[4])
assert np.isfinite(fin[feat_cols].iloc[7:].to_numpy()).all()
print('C1 VALIDÉ — features:',feat_cols)
print(fin[['date']+feat_cols+['y_next_oc_pct']].iloc[6:9].round(4).to_string(index=False))

### Mission · C2 — Construit le tenseur 3D et ses dates de maturité

**Contrat :** `fenetres_financieres(df,feat_cols,L)` retourne `X(N,L,F)`, `y(N,1)`, `decision_idx(N,)`, `label_idx(N,)`. Le tableau `df` contient déjà des NaN sur le préchauffage ; commencer à la première ligne où toutes les features sont finies. Arrêter avant la cible manquante. Pour chaque fenêtre, les temps doivent être **consécutifs** ; `label_idx = decision_idx + 1` selon les indices du tableau initial. Ne pas convertir la cible en caractéristique.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def fenetres_financieres(df,feat_cols,L):
    raise NotImplementedError('C2 : créez les tenseurs de features et les indices de dates des cibles')

In [ ]:
L_fin=16
Xfin_raw,yfin,dec_fin,maturity_fin=fenetres_financieres(fin,feat_cols,L_fin)
assert Xfin_raw.ndim==3 and Xfin_raw.shape[1:]==(16,5)
assert yfin.shape==(len(Xfin_raw),1)
assert np.array_equal(maturity_fin,dec_fin+1)
assert np.isclose(yfin[0,0],fin['y_next_oc_pct'].iloc[dec_fin[0]])
cut_val_fin=int(.60*len(fin));cut_test_fin=int(.80*len(fin))
itr_f,iva_f,ite_f=split_indices(dec_fin,1,cut_val_fin,cut_test_fin)
assert max(maturity_fin[itr_f])<cut_val_fin
assert max(maturity_fin[iva_f])<cut_test_fin
assert len(itr_f)>200 and len(iva_f)>50 and len(ite_f)>50
scaler_fin=StandardScaler().fit(Xfin_raw[itr_f].reshape(-1,len(feat_cols)))
Xfin=scaler_fin.transform(Xfin_raw.reshape(-1,len(feat_cols))).reshape(Xfin_raw.shape).astype('float32')
print('C2 VALIDÉ :',(len(itr_f),len(iva_f),len(ite_f)),'| MLP',Xfin[itr_f].reshape(len(itr_f),-1).shape,'| LSTM',Xfin[itr_f].shape)

## C3 — Choisir et entraîner votre architecture multivariée

Vous connaissez les modèles du parcours univarié. **N'en choisissez pas un à l'avance pour sa réputation** : comparez une baseline `zéro`, Ridge et *au moins deux* architectures neuronales, dont une architecture à mémoire.

Gardez la même cible `y_next_oc_pct`, la même fenêtre, les mêmes dates et un budget CPU raisonnable. Sélectionnez la plus petite **MAE VAL**. Toute consultation des cibles TEST avant ce choix invalide le protocole.

### Mission · C3 — Évaluez vos modèles financiers sur VALIDATION

**À construire :** régression Ridge (`alpha=10`), baseline zéro, MLP et au moins deux modèles parmi RNN/GRU/LSTM sur données normalisées. Conserver `models_fin`, `pred_val_fin`, `scores_val_fin` et `chosen_fin`, puis produire un tableau VALIDATION. Choisir uniquement d'après la MAE de validation. Le TEST reste fermé jusqu'au bloc suivant.

**Petit budget** : MLP, GRU, LSTM avec `hidden=10` et `epochs=16`, ou plus si CPU le permet.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
models_fin={};pred_val_fin={};scores_val_fin={}
# TODO : comparer les références et au moins deux réseaux, puis définir chosen_fin.
raise NotImplementedError('C3 : construisez les expériences et figez chosen_fin sur VAL')

In [ ]:
assert len(scores_val_fin)>=4 and chosen_fin in scores_val_fin
assert {'Zéro','Ridge'}.issubset(scores_val_fin)
assert any(k in scores_val_fin for k in ('GRU','LSTM','RNN'))
assert all(np.isfinite(row['MAE']) for row in scores_val_fin.values())
print('C3 VALIDÉ — modèle figé:',chosen_fin)

### Ouverture contrôlée de TEST
Dès que `chosen_fin` est figé, exécutez cette cellule **une fois**. Ne modifiez plus le choix en fonction de TEST. Le résultat porte sur une réalisation fictive et n'est ni un conseil d'investissement ni une preuve de rentabilité réelle.

In [ ]:
if chosen_fin=='Zéro':pred_test_fin=np.zeros(len(ite_f))
elif chosen_fin=='Ridge':pred_test_fin=ridge_fin.predict(Xfin[ite_f].reshape(len(ite_f),-1))
else:pred_test_fin=predict_numpy(models_fin[chosen_fin]['model'],Xfin[ite_f]).reshape(-1)
ytest_fin=yfin[ite_f].reshape(-1)
print('C3 — scores TEST du modèle gelé',chosen_fin,metrics(ytest_fin,pred_test_fin))
plt.scatter(ytest_fin,pred_test_fin,s=12,alpha=.5);plt.xlabel('Rendement intrajournalier réalisé t+1 (%)');plt.ylabel('Prévision faite après close t (%)');plt.title('TEST — modèle figé sur validation');plt.show()

### Mission · C4 — Convertissez les prédictions en positions et gains après coûts

**Contrat :** `simulation_signal(predictions, rendements_pct, seuil=.12,cout_aller_retour_pct=.12)` retourne un dictionnaire `position`, `net_day`, `equity` :
- position +1 si prévision `> seuil`, −1 si `< −seuil`, 0 sinon ;
- `net_day = position × (exp(r_intraday_log_pct/100)-1) − (cout/100)×abs(position)` ;
- `equity = cumprod(1+net_day)`, capital initial 1.

L'ordre chronologique est obligatoire : décision **après clôture t**, position de **l'ouverture à la clôture t+1**. Hypothèse didactique : aller-retour intrajournalier, absence de levier, slippage intégré forfaitairement au coût ; ne pas exploiter `close[t]→open[t+1]`. Le seuil est fixé ex ante, jamais choisi sur TEST.

**Cellule à compléter.** Remplacez `raise NotImplementedError(...)` par votre solution ; les auto-tests situés juste après vous guideront. Aucun code solution ne se trouve dans l’édition STUDENT.

In [ ]:
def simulation_signal(predictions,rendements_pct,seuil=.12,cout_aller_retour_pct=.12):
    raise NotImplementedError('C4 : formez la position et le P&L net, puis le capital composé')

In [ ]:
toy=simulation_signal([.4,-.5,.01],[.5,-.2,1.0],seuil=.12,cout_aller_retour_pct=.12)
assert np.array_equal(toy['position'],[1,-1,0])
assert np.isclose(toy['net_day'][0],np.expm1(.5/100)-.0012)
assert np.isclose(toy['net_day'][1],-np.expm1(-.2/100)-.0012)
assert toy['net_day'][2]==0 and len(toy['equity'])==3
print('C4 VALIDÉ — coût et horloge corrects')
trade=simulation_signal(pred_test_fin,ytest_fin,seuil=.12,cout_aller_retour_pct=.12)
print('Nombre de séances actives:',np.count_nonzero(trade['position']))
print('Capital final fictif:',round(float(trade['equity'][-1]),4),'| rendement net composé:',round(100*(trade['equity'][-1]-1),2),'%')
plt.plot(trade['equity']);plt.axhline(1,color='black',lw=.8);plt.title('Capital simulé après coût forfaitaire — données fictives');plt.xlabel('Séances TEST');plt.ylabel('Capital initial = 1');plt.show()

# Dossier final à remettre (quatre livrables)

**D1 — Modélisation univariée.** Un tableau avec `T`, `L`, `h`, `N`, `X`, `y` ; démonstration de maturité des labels aux frontières ; baseline et premier MLP.

**D2 — Réseaux et comparaison.** Schémas `MLP/RNN/GRU/LSTM` et dimensions de toutes les matrices ; courbes TRAIN/VAL ; tableau comparatif ; justification de la sélection ; effet d'au moins un hyperparamètre.

**D3 — Extension multivariée.** Justification de cinq caractéristiques causales, tableau de shapes `MLP (B,L×F)` vs `LSTM (B,L,F)` ; tableau validation ; évaluation TEST unique.

**D4 — Trading et analyse critique.** Tracer le capital net, mentionner seuil et coût en unités cohérentes, expliciter l'exécution `close[t]`→`open[t+1]` et expliquer pourquoi qualité prédictive ≠ rentabilité.

**Critères de validation :** A1–A4, B1–B3, C1–C4 exécutés, aucune fuite, visualisations légendées, code propre. À la remise, joindre le `.ipynb` et un court compte rendu (4–6 pages max).

**À ne pas faire :** sélectionner sur TEST, ajuster `StandardScaler` sur tout le dataset, utiliser `y_next_oc_pct` comme feature, changer le seuil après la lecture du rendement TEST, conclure que l'on a découvert une stratégie rentable réelle.